# Lab: k-NN & Decision Tree on the Heart Attack Dataset

**Goal:** Train and compare **k-NN** and **Decision Tree** classifiers for predicting heart attack risk (binary classification).  
You will:
1. Load `heart.csv` (Heart Attack dataset)
2. Clean & preprocess data
3. Tune hyperparameters with Cross-Validation (CV)
4. Train final models and evaluate on a held-out test set
5. Make predictions on new samples



In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

RANDOM_STATE = 42


## 1) Load dataset

In [ ]:
# Load dataset
CSV_PATH = 'Heart_Attack.csv' #<<<<<<<<<<<<<<<<<
df = pd.read_csv(CSV_PATH)    #<<<<<<<<<<<<<<<<<
# ---------------------

# Display basic info
print("Loaded:", CSV_PATH)    #<<<<<<<<<<<<<<<<<
print("Shape:", df.shape)
display(df.head())

Loaded: Heart_Attack.csv
Shape: (303, 14)


,age,sex,cp,trtbps,chol,fbs,restecg,thalachh,exng,oldpeak,slp,caa,thall,output
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


## 2) Inspect & basic cleaning

In [ ]:
print("Columns:", list(df.columns))
print("\nMissing values per column:")
print(df.isna().sum())

# Basic cleanup (drop rows with missing values)
df = df.dropna().reset_index(drop=True)
print("\nAfter dropna -> Shape:", df.shape)

print("Using TARGET_COL =",  'output')  #<<<<<<<<<<<

# Separate X / y
X = df.drop(columns=['output'])  #<<<<<<<<<<<
y = df['output']                 #<<<<<<<<<<<

print("X shape:", X.shape, "y counts:", np.bincount(y))


Columns: ['age', 'sex', 'cp', 'trtbps', 'chol', 'fbs', 'restecg', 'thalachh', 'exng', 'oldpeak', 'slp', 'caa', 'thall', 'output']

Missing values per column:
age         0
sex         0
cp          0
trtbps      0
chol        0
fbs         0
restecg     0
thalachh    0
exng        0
oldpeak     0
slp         0
caa         0
thall       0
output      0
dtype: int64

After dropna -> Shape: (303, 14)
Using TARGET_COL = output
X shape: (303, 13) y counts: [138 165]


## 3) Train/Test split  
We keep the test set untouched until the final evaluation.

In [5]:
X_train, X_test, y_train, y_test =  train_test_split(X, y, test_size=0.25, random_state=7, stratify=y) #<<<<<<<<<<<<<<<<<

print("Train:", X_train.shape, "Test:", X_test.shape)


Train: (227, 13) Test: (76, 13)


## 4) k-NN (with scaling) + CV to select best `k`

### 4.1) Scaling
**Why scaling?** k-NN uses distances, so features must be on comparable scales.


In [6]:
# ---- scaling with StandardScaler ----
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("After scaling:")
print("\nX_train_s mean:", np.mean(X_train_s, axis=0))
print("\nX_train_s max:", np.max(X_train_s, axis=0))
print("\nX_train_s min:", np.min(X_train_s, axis=0))


After scaling:

X_train_s mean: [-2.50411537e-16 -9.78170066e-18 -6.35810543e-17  1.97590353e-16
 -2.93451020e-17 -7.82536053e-17  1.17380408e-17 -3.75617305e-16
 -3.13014421e-17 -1.33031129e-16  9.39043263e-17  2.73887618e-17
 -1.38411064e-16]

X_train_s max: [2.42012019 0.69545678 1.90480269 3.81329829 3.71071633 2.16217483
 2.68744229 2.25313087 1.46723474 4.46166531 0.94086474 3.11399376
 1.11469898]

X_train_s min: [-2.73671271 -1.43790388 -0.97140714 -2.07980277 -2.4814976  -0.46249729
 -1.10168442 -3.43893885 -0.6815542  -0.8711141  -2.31984206 -0.74345008
 -3.65957779]


### 4.2) Cross-validation to select best k value

In [ ]:
# Initialize StratifiedKFold (5 folds)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)      #<<<<<<<<<<<

In [ ]:
# Make a list of k values to try
# CV to try k = 1, 3, 5, ..., 15
k_list = list(range(1, 16, 2))         #<<<<<<<<<<<

# Build mean_scores list
mean_scores = []

In [9]:
# Loop over k values and perform CV
# For each k, create KNN model and get CV accuracy
for k in k_list:
    knn = KNeighborsClassifier(n_neighbors=k)                                    #<<<<<<<<<<<
    scores = cross_val_score(knn, X_train_s, y_train, cv=cv, scoring='accuracy') #<<<<<<<<<<<
    mean_scores.append(scores.mean())

# Find best k
best_idx = int(np.argmax(mean_scores))
best_k = k_list[best_idx]
best_cv_acc = mean_scores[best_idx]

In [10]:
# Print results
print("=== k-NN CV results (TRAIN only) ===")
for k, m in zip(k_list, mean_scores):
    print(f"k={k:2d}  mean_CV_acc={m:.4f}")

print(f"\nBest k = {best_k} (mean CV acc = {best_cv_acc:.4f})")

=== k-NN CV results (TRAIN only) ===
k= 1  mean_CV_acc=0.7713
k= 3  mean_CV_acc=0.8198
k= 5  mean_CV_acc=0.8373
k= 7  mean_CV_acc=0.8241
k= 9  mean_CV_acc=0.8242
k=11  mean_CV_acc=0.8154
k=13  mean_CV_acc=0.8155
k=15  mean_CV_acc=0.8110

Best k = 5 (mean CV acc = 0.8373)


### 4.3) Train final k-NN model with best k

In [ ]:
# Train final k-NN model with best k
best_knn = KNeighborsClassifier(n_neighbors=best_k)         #<<<<<<<<<<<<
best_knn.fit(X_train_s, y_train)                            #<<<<<<<<<<<<

print(f"Final model trained successfully with k={best_k}")  #<<<<<<<<<<<<

Final model trained successfully with k=5


### 4.4) Evaluate k-NN on test set

In [ ]:
# Evaluate k-NN on test set
y_pred_knn = best_knn.predict(X_test_s)                    #<<<<<<<<<<<<<
acc_knn = accuracy_score(y_test, y_pred_knn)              #<<<<<<<<<<<<<
cm_knn = confusion_matrix(y_test, y_pred_knn)              #<<<<<<<<<<<<<


In [15]:
# Print results
print("\n=== k-NN Test ===")
print("Accuracy:", round(acc_knn, 4))
print("Confusion matrix [[TN FP],[FN TP]]:\n", cm_knn)
print("\nReport:\n", classification_report(y_test, y_pred_knn, digits=4))


=== k-NN Test ===
Accuracy: 0.8289
Confusion matrix [[TN FP],[FN TP]]:
 [[27  8]
 [ 5 36]]

Report:
               precision    recall  f1-score   support

           0     0.8438    0.7714    0.8060        35
           1     0.8182    0.8780    0.8471        41

    accuracy                         0.8289        76
   macro avg     0.8310    0.8247    0.8265        76
weighted avg     0.8300    0.8289    0.8281        76



## 5) Decision Tree + CV to select best `max_depth`

**Note:** Decision trees do **not** require scaling.

#

### 5.1) Cross-validation to select best max_depth

In [ ]:
# Initialize StratifiedKFold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)      #<<<<<<<<<<<<<

In [ ]:
# Create a list of max_depth values to try
# max_depth => 1 to 10
depth_list = list(range(1, 11))          #<<<<<<<<<<<<<
 
# Build mean_scores list
mean_scores = []

In [ ]:
# Loop over depth values and perform CV
for d in depth_list:
    tree = DecisionTreeClassifier(max_depth=d, random_state=42)                    #<<<<<<<<<<<<<
    scores = cross_val_score(tree, X_train, y_train, cv=cv, scoring='accuracy')    #<<<<<<<<<<<<<
    mean_scores.append(scores.mean())

In [ ]:
# Find best max_depth
best_idx = int(np.argmax(mean_scores))           
best_depth = depth_list[best_idx] 
best_cv_acc = mean_scores[best_idx]

In [20]:
# Print results
print("=== Decision Tree CV results (TRAIN only) ===")
for d, m in zip(depth_list, mean_scores):
    d_text = "None" if d is None else str(d)
    print(f"max_depth={d_text:>4s}  mean_CV_acc={m:.4f}")

print(f"\nBest max_depth = {('None' if best_depth is None else best_depth)} (mean CV acc = {best_cv_acc:.4f})")

=== Decision Tree CV results (TRAIN only) ===
max_depth=   1  mean_CV_acc=0.7399
max_depth=   2  mean_CV_acc=0.7797
max_depth=   3  mean_CV_acc=0.8196
max_depth=   4  mean_CV_acc=0.8065
max_depth=   5  mean_CV_acc=0.7755
max_depth=   6  mean_CV_acc=0.7842
max_depth=   7  mean_CV_acc=0.7665
max_depth=   8  mean_CV_acc=0.7666
max_depth=   9  mean_CV_acc=0.7666
max_depth=  10  mean_CV_acc=0.7666

Best max_depth = 3 (mean CV acc = 0.8196)


### 5.2) Train final Decision Tree model with best max_depth

In [21]:
# Train final Decision Tree model with best max_depth
best_tree = DecisionTreeClassifier(max_depth=best_depth, random_state=42)       #<<<<<<<<<<<<<
best_tree.fit(X_train, y_train)                                                 #<<<<<<<<<<<<<

DecisionTreeClassifier(max_depth=3, random_state=42)

In [ ]:
# Evaluate Decision Tree on test set
y_pred_tree = best_tree.predict(X_test)            #<<<<<<<<<<<<<
acc_tree = accuracy_score(y_test, y_pred_tree)     #<<<<<<<<<<<<<
cm_tree = confusion_matrix(y_test, y_pred_tree)    #<<<<<<<<<<<<<

In [23]:
# Print results
print("\n=== Decision Tree Test ===")
print("Accuracy:", round(acc_tree, 4))
print("Confusion matrix [[TN FP],[FN TP]]:\n", cm_tree)
print("\nReport:\n", classification_report(y_test, y_pred_tree, digits=4))



=== Decision Tree Test ===
Accuracy: 0.7237
Confusion matrix [[TN FP],[FN TP]]:
 [[24 11]
 [10 31]]

Report:
               precision    recall  f1-score   support

           0     0.7059    0.6857    0.6957        35
           1     0.7381    0.7561    0.7470        41

    accuracy                         0.7237        76
   macro avg     0.7220    0.7209    0.7213        76
weighted avg     0.7233    0.7237    0.7233        76



## 6) Compare models (test set)

You can report:
- accuracy
- confusion matrix
- precision/recall/F1 for class 1 (heart attack)


In [ ]:
results = pd.DataFrame([
    {"model": "k-NN", "test_accuracy": acc_knn},                 #<<<<<<<<<<<<<
    {"model": "Decision Tree", "test_accuracy": acc_tree},       #<<<<<<<<<<<<<
]).sort_values("test_accuracy", ascending=False)

display(results)


,model,test_accuracy
0,k-NN,0.828947
1,Decision Tree,0.723684


## 7) Predict a new patient

You must provide values for **all feature columns** exactly as in `X.columns`.


In [26]:
print("Feature columns:", list(X.columns))

# Example input 
new_sample = {col: float(X[col].median()) for col in X.columns}  # simple default using median

new_df = pd.DataFrame([new_sample])
display(new_df)

# k-NN needs scaling
new_df_s = scaler.transform(new_df)

pred_knn = int(best_knn.predict(new_df_s)[0])
pred_tree = int(best_tree.predict(new_df)[0])

print("k-NN prediction (0/1):", pred_knn)
print("Decision Tree prediction (0/1):", pred_tree)


Feature columns: ['age', 'sex', 'cp', 'trtbps', 'chol', 'fbs', 'restecg', 'thalachh', 'exng', 'oldpeak', 'slp', 'caa', 'thall']


,age,sex,cp,trtbps,chol,fbs,restecg,thalachh,exng,oldpeak,slp,caa,thall
0,55.0,1.0,1.0,130.0,240.0,0.0,1.0,153.0,0.0,0.8,1.0,0.0,2.0


k-NN prediction (0/1): 1
Decision Tree prediction (0/1): 1
